In [1]:
import sys, time, re, json, pathlib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)
GENES = ["CYP2C19", "CYP2C9", "VKORC1", "DPYD", "TPMT"]
MV = "https://myvariant.info/v1"
AVI_FILE = DATA / "alphagenome_avi.tsv"

ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

FIELDS = ",".join([
    "dbsnp.rsid",                        # the rs number
    "snpeff.ann.effect",                 # consequence term
    "cadd.phred",                        # CADD
    "dbnsfp.revel.score",                # REVEL
    "dbnsfp.alphamissense",              # AlphaMissense
    "dbnsfp.aa.ref", "dbnsfp.aa.alt", "dbnsfp.aa.pos", "dbnsfp.genename",
    "gnomad_genome.af.af", "gnomad_exome.af.af",
    "clinvar.rcv.clinical_significance",
    "clinvar.rcv.review_status",
    "clinvar.gene.symbol",
    "clinvar.variant_id",                # join key for CADD (see Section 3)
])

def first(x):
    """Some fields come back as a single value, some as a list. Take the first."""
    return (x[0] if x else None) if isinstance(x, list) else x

def dig(d, *path):
    """Walk a nested dict safely. Returns None if any level is missing."""
    for k in path:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
        if d is None:
            return None
    return d

# ClinVar review status, as the familiar star rating
STARS = {"practice guideline": 4,
         "reviewed by expert panel": 3,
         "criteria provided, multiple submitters, no conflicts": 2,
         "criteria provided, conflicting classifications": 1,
         "criteria provided, conflicting interpretations": 1,
         "criteria provided, single submitter": 1,
         "no assertion criteria provided": 0,
         "no classification provided": 0}

# snpEff reports one effect per transcript. Keep the most severe one.
SEVERITY = ["stop_gained", "frameshift_variant", "splice_acceptor_variant",
            "splice_donor_variant", "start_lost", "stop_lost", "missense_variant",
            "splice_region_variant", "synonymous_variant", "5_prime_UTR_variant",
            "3_prime_UTR_variant", "intron_variant", "upstream_gene_variant",
            "downstream_gene_variant", "intergenic_region"]

def all_effects(h):
    """Every effect snpEff reported, across every transcript, split into single terms."""
    ann = dig(h, "snpeff", "ann")
    raw = [a.get("effect") for a in (ann if isinstance(ann, list) else [ann]) if isinstance(a, dict)]
    terms = []
    for e in raw:                       # snpEff emits compound terms joined by "&"
        if e:
            terms += [t for t in e.split("&") if t != "structural_interaction_variant"]
    return sorted(set(terms))

def worst_effect(h):
    """Collapse to ONE term: the most severe across all transcripts."""
    terms = all_effects(h)
    if not terms:
        return None
    return min(terms, key=lambda t: SEVERITY.index(t) if t in SEVERITY else 99)

def to_row(h, gene=None):
    aaref, aaalt = first(dig(h, "dbnsfp", "aa", "ref")), first(dig(h, "dbnsfp", "aa", "alt"))
    aapos = first(dig(h, "dbnsfp", "aa", "pos"))
    rcv = first(dig(h, "clinvar", "rcv")) or {}   # one record per submission; take the first
    rev = rcv.get("review_status")
    am = dig(h, "dbnsfp", "alphamissense", "score")
    return {
        "hgvs_g":        h.get("_id"),
        "rsid":          dig(h, "dbsnp", "rsid"),
        "gene":          first(dig(h, "clinvar", "gene", "symbol")) or first(dig(h, "dbnsfp", "genename")) or gene,
        "clinvar_id":    dig(h, "clinvar", "variant_id"),
        "consequence":   worst_effect(h),
        "consequence_all": ";".join(all_effects(h)),   # keep the full picture
        "aa_change":     f"{aaref}{aapos}{aaalt}" if (aaref and aaalt and aapos) else None,
        "cadd_phred":    dig(h, "cadd", "phred"),
        "revel":         first(dig(h, "dbnsfp", "revel", "score")),
        "alphamissense": first(am) if isinstance(am, list) else am,
        "gnomad_af":     dig(h, "gnomad_genome", "af", "af") or dig(h, "gnomad_exome", "af", "af"),
        "clinvar_sig":   rcv.get("clinical_significance"),
        "clinvar_stars": STARS.get(str(rev).lower(), 0),
    }

FORCE_REFETCH = False

# Known pharmacogenomic alleles, so we can check the fetch worked and
# recognise them later.
ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

def fetch_gene(gene, verbose=True):
    """Search ClinVar for one gene, annotate every hit, return a tidy DataFrame."""
    r = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg38",
        "fields": "_id", "size": 1000})
    r.raise_for_status()
    ids = [h["_id"] for h in r.json().get("hits", [])]
    if verbose:
        print(f"  {gene:9s} {len(ids):>5} variants found", end="", flush=True)

    rows = []
    for i in range(0, len(ids), 500):
        rr = requests.post(f"{MV}/variant", timeout=180,
                           data={"ids": ",".join(ids[i:i+500]),
                                 "fields": FIELDS, "assembly": "hg38"})
        rr.raise_for_status()
        rows += [to_row(h, gene) for h in rr.json()]
        time.sleep(0.3)                        # be polite to a free API

    df = pd.DataFrame(rows).drop_duplicates("hgvs_g")

    # MyVariant.info only carries genome-wide CADD on hg19. Ask the same
    # question on hg19 and join on the ClinVar variant ID, which is build-free.
    r19 = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg19",
        "fields": "clinvar.variant_id,cadd.phred", "size": 1000})
    r19.raise_for_status()
    hits19 = r19.json().get("hits", [])
    cadd = {}
    for h in hits19:
        vid = first(dig(h, "clinvar", "variant_id"))      # occasionally a list
        ph = dig(h, "cadd", "phred")
        if vid is not None and ph is not None:
            cadd[vid] = ph
    df["cadd_phred"] = df.clinvar_id.map(cadd)

    if len(ids) >= 1000 or len(hits19) >= 1000:
        print(f"\n  [!] {gene} hit the 1000-result cap. The two builds may have been")
        print( "      truncated to different sets, so some CADD scores will be missing.")
    if verbose:
        print(f"  ->  {len(df)} annotated")
    return df

def numeric(df):
    """Force the score columns to numbers. Strings here break everything later."""
    for c in ["cadd_phred", "revel", "alphamissense", "spliceai_dg",
              "spliceai_dl", "avi", "gnomad_af", "clinvar_stars"]:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def clinvar_label(sig):
    if not isinstance(sig, str): return "none"
    s = sig.lower()
    if "conflict" in s: return "conflicting"
    if "pathogenic" in s and "non-pathogenic" not in s: return "P/LP"
    if "benign" in s: return "B/LB"
    if "uncertain" in s: return "VUS"
    return "other"          # includes "drug response"

STRENGTH = [(18.7, "strong, pathogenic"), (4.3, "moderate, pathogenic"),
            (2.08, "supporting, pathogenic"), (0.48, "no evidence either way"),
            (0.23, "supporting, benign"), (0.053, "moderate, benign"),
            (0.0, "strong, benign")]

def strength_for(lr):
    if lr != lr: return "undefined"
    for cut, name in STRENGTH:
        if lr >= cut: return name
    return "strong, benign"

def calibrate(df, score_col, bands):
    d = df[df.label.isin(["P/LP", "B/LB"]) & df[score_col].notna()].copy()
    d["band"] = pd.cut(d[score_col], bands)
    n_p, n_b = (d.label == "P/LP").sum(), (d.label == "B/LB").sum()
    g = d.groupby("band", observed=False)
    out = pd.DataFrame({"n": g.size(),
                        "n_PLP": g.label.apply(lambda s: (s == "P/LP").sum()),
                        "n_BLB": g.label.apply(lambda s: (s == "B/LB").sum())})
    with np.errstate(divide="ignore", invalid="ignore"):
        out["LR+"] = ((out.n_PLP / max(n_p, 1)) / (out.n_BLB / max(n_b, 1))).round(2)
    out["evidence"] = out["LR+"].map(strength_for)
    out.attrs["n_p"], out.attrs["n_b"] = n_p, n_b
    return out

print("helpers loaded:", ", ".join(["fetch_gene", "numeric", "to_row", "clinvar_label",
                                    "calibrate", "strength_for", "worst_effect"]))

/opt/anaconda3/envs/biot6900/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


helpers loaded: fetch_gene, numeric, to_row, clinvar_label, calibrate, strength_for, worst_effect


In [2]:
MY_GENE = "CYP2C9"        # <-- set this to YOUR assigned gene
assert MY_GENE in GENES, f"MY_GENE must be one of {GENES}"
MY_FILE = DATA / f"{MY_GENE}.tsv"
print("working on", MY_GENE)

working on CYP2C9


In [ ]:
# Q1
if MY_FILE.exists() and not FORCE_REFETCH:
    G = pd.read_csv(MY_FILE, sep="\t")
else:
    G = fetch_gene(MY_GENE)
    G.to_csv(MY_FILE, sep="\t", index=False)

G = numeric(G)
G["label"] = G.clinvar_sig.map(clinvar_label)

print(f"\n[1] Count: {len(G)} variants in {MY_GENE}")
print(G.label.value_counts().to_string())

print("\n[2] Anchors for this gene:")
mine = {rs: n for rs, n in ANCHORS.items() if n.startswith(MY_GENE)}
for rs, name in mine.items():
    hit = G[G.rsid == rs]
    print(f"  {name:12s} {rs:11s}", "FOUND" if len(hit) else "MISSING -> fix the fetch")
    if len(hit):
        print(hit[["rsid", "consequence", "cadd_phred", "revel", "gnomad_af", "clinvar_sig"]].to_string(index=False))

print("\n[3] Score columns populated (non-null %):")
for c in ["cadd_phred", "revel", "alphamissense", "gnomad_af"]:
    print(f"  {c:14s} {G[c].notna().sum():>5}  ({100*G[c].notna().mean():.1f}%)")

  CYP2C9       41 variants found  ->  41 annotated

[1] Count: 41 variants in CYP2C9
label
VUS      22
other    10
B/LB      8
P/LP      1

[2] Anchors for this gene:
  CYP2C9*2     rs1799853   FOUND
     rsid      consequence  cadd_phred  revel  gnomad_af   clinvar_sig
rs1799853 missense_variant        29.1   0.34   0.088164 drug response
  CYP2C9*3     rs1057910   FOUND
     rsid      consequence  cadd_phred  revel  gnomad_af   clinvar_sig
rs1057910 missense_variant        20.4   0.07   0.049454 drug response

[3] Score columns populated (non-null %):
  cadd_phred        38  (92.7%)
  revel             33  (80.5%)
  alphamissense     33  (80.5%)
  gnomad_af         29  (70.7%)


In [ ]:
# Q2
TOOLS = ["cadd_phred", "revel", "alphamissense"]

# keep variants with no consequence term visible instead of letting groupby drop them
cons = G.consequence.fillna("no_annotation")

# (a) per consequence class: how many variants, and how many each tool scored
tab = G[TOOLS].notna().groupby(cons).sum().astype(int)
tab.insert(0, "n_variants", cons.value_counts())
tab = tab.sort_values("n_variants", ascending=False)

# (b) the table and the overall percentage scored by each tool
print("variants scored, by consequence class\n")
print(tab.to_string())
print(f"\ntotal variants: {len(G)}")
print("\noverall % scored by each tool")
print((G[TOOLS].notna().mean() * 100).round(1).to_string())

# (c) classes with zero REVEL scores
zero = tab[tab.revel == 0]
print("\nconsequence classes with ZERO REVEL scores:")
if len(zero):
    for c, r in zero.iterrows():
        print(f"  {c:28s} n={r.n_variants}  cadd={r.cadd_phred}  alphamissense={r.alphamissense}")
else:
    print("  none")
print("\nA blank means the tool could not score the variant. It does not mean benign.")

variants scored, by consequence class

                    n_variants  cadd_phred  revel  alphamissense
consequence                                                     
missense_variant            33          33     33             33
synonymous_variant           3           3      0              0
intron_variant               2           2      0              0
no_annotation                2           0      0              0
frameshift_variant           1           0      0              0

total variants: 41

overall % scored by each tool
cadd_phred       92.7
revel            80.5
alphamissense    80.5

consequence classes with ZERO REVEL scores:
  synonymous_variant           n=3  cadd=3  alphamissense=0
  intron_variant               n=2  cadd=2  alphamissense=0
  no_annotation                n=2  cadd=0  alphamissense=0
  frameshift_variant           n=1  cadd=0  alphamissense=0

A blank means the tool could not score the variant. It does not mean benign.


In [14]:
# Q3
MAX_AF = 0.15

# keep an untouched copy so re-running this cell never filters twice
if "G_full" not in globals():
    G_full = G.copy()
src = G_full

def apply_af(df, cutoff):
    # no gnomAD entry means absent from the database, not "common", so it is kept
    return df[df.gnomad_af.isna() | (df.gnomad_af <= cutoff)]

def interesting(df):
    sig = df.clinvar_sig.fillna("").str.lower()
    return df[(df.label == "P/LP") | sig.str.contains("drug response")]

cols = ["rsid", "aa_change", "consequence", "gnomad_af", "clinvar_sig", "label"]

# (b) how many variants the chosen filter removes
kept = apply_af(src, MAX_AF)
removed = src.drop(kept.index)
print(f"MAX_AF = {MAX_AF}")
print(f"variants before: {len(src)}   removed: {len(removed)}   kept: {len(kept)}")
print(f"kept with no gnomAD value: {int(kept.gnomad_af.isna().sum())}")

# (c) P/LP or drug response variants removed by the chosen filter
lost = interesting(removed)
print(f"\nP/LP or drug-response variants removed by MAX_AF = {MAX_AF}: {len(lost)}")
if len(lost):
    print(lost[cols].to_string(index=False))

# anchor check: known functional alleles for this gene must survive
mine = {rs: n for rs, n in ANCHORS.items() if n.startswith(MY_GENE)}
kept_rsids = set(kept.rsid)
for rs, name in mine.items():
    row = src[src.rsid == rs]
    af = row.gnomad_af.iloc[0] if len(row) else float("nan")
    status = "KEPT" if rs in kept_rsids else "REMOVED, change threshold or justify"
    print(f"  anchor {name:12s} {rs:11s} gnomad_af={af}  ->  {status}")

# (d) contrast: a rare-disease filter at 0.001
kept_rd = apply_af(src, 0.001)
removed_rd = src.drop(kept_rd.index)
lost_rd = interesting(removed_rd)
print(f"\nCONTRAST, rare-disease filter at 0.001: removes {len(removed_rd)} of {len(src)} variants")
print(f"P/LP or drug-response variants it removes: {len(lost_rd)}")
if len(lost_rd):
    show = lost_rd[cols].sort_values("gnomad_af", ascending=False)
    print(show.to_string(index=False))

# (e) continue with the filtered set
G = kept.copy()
print(f"\nG is now the filtered set: {len(G)} variants")

MAX_AF = 0.15
variants before: 41   removed: 2   kept: 39
kept with no gnomAD value: 12

P/LP or drug-response variants removed by MAX_AF = 0.15: 0
  anchor CYP2C9*2     rs1799853   gnomad_af=0.0881641  ->  KEPT
  anchor CYP2C9*3     rs1057910   gnomad_af=0.0494537  ->  KEPT

CONTRAST, rare-disease filter at 0.001: removes 10 of 41 variants
P/LP or drug-response variants it removes: 4
      rsid aa_change      consequence  gnomad_af   clinvar_sig label
 rs1799853     R144C missense_variant   0.088164 drug response other
 rs1057910     I359L missense_variant   0.049454 drug response other
 rs7900194     R150H missense_variant   0.015809 drug response other
rs28371685     R335W missense_variant   0.007015 drug response other

G is now the filtered set: 39 variants


In [15]:
# Q4
SCORE_COL = "cadd_phred"
BANDS     = [0, 10, 20, 25, 30, 50]

MIN_PER_CLASS = 5    # my own rule: need at least 5 P/LP and 5 B/LB to calibrate at all
MIN_PER_BAND  = 10   # the assignment's flag for a fragile band

# (a) calibrate on the filtered set
cal = calibrate(G, SCORE_COL, BANDS)
n_p, n_b = cal.attrs["n_p"], cal.attrs["n_b"]

# (b) what the calibration rests on
print(f"score column: {SCORE_COL}")
print(f"labelled variants with a score: {n_p + n_b}  (P/LP = {n_p}, B/LB = {n_b})")
print(f"labelled variants in G before requiring a score: "
      f"P/LP = {(G.label == 'P/LP').sum()}, B/LB = {(G.label == 'B/LB').sum()}")
print("\n" + cal.to_string())

# (c) fragile bands
thin = cal[cal.n < MIN_PER_BAND]
print(f"\nbands resting on fewer than {MIN_PER_BAND} variants: {len(thin)} of {len(cal)}")
for band, r in thin.iterrows():
    print(f"  {str(band):10s} n={r.n}  (P/LP={r.n_PLP}, B/LB={r.n_BLB})  LR+={r['LR+']}")

# (d) is calibration possible at all?
CALIBRATED = (n_p >= MIN_PER_CLASS) and (n_b >= MIN_PER_CLASS) and (len(thin) < len(cal))
if CALIBRATED:
    print("\nCALIBRATION USABLE: report LRs together with their denominators.")
else:
    print(f"\nCALIBRATION NOT POSSIBLE for {MY_GENE}.")
    print(f"  Only {n_p} P/LP and {n_b} B/LB variants carry a {SCORE_COL}.")
    print("  A likelihood ratio needs both classes in every band. With these counts the")
    print("  LR+ values above are noise: an empty B/LB cell gives an infinite ratio that")
    print("  strength_for() would call 'strong, pathogenic'. Do NOT use the table above.")
    print("  Route taken: state that calibration is not possible (no pooling).")
    print("  Every VUS therefore stays a VUS in Q5, because no evidence has been established.")

score column: cadd_phred
labelled variants with a score: 8  (P/LP = 1, B/LB = 7)
labelled variants in G before requiring a score: P/LP = 1, B/LB = 8

          n  n_PLP  n_BLB   LR+                evidence
band                                                   
(0, 10]   4      1      3  2.33  supporting, pathogenic
(10, 20]  1      0      1  0.00          strong, benign
(20, 25]  3      0      3  0.00          strong, benign
(25, 30]  0      0      0   NaN               undefined
(30, 50]  0      0      0   NaN               undefined

bands resting on fewer than 10 variants: 5 of 5
  (0, 10]    n=4  (P/LP=1, B/LB=3)  LR+=2.33
  (10, 20]   n=1  (P/LP=0, B/LB=1)  LR+=0.0
  (20, 25]   n=3  (P/LP=0, B/LB=3)  LR+=0.0
  (25, 30]   n=0  (P/LP=0, B/LB=0)  LR+=nan
  (30, 50]   n=0  (P/LP=0, B/LB=0)  LR+=nan

CALIBRATION NOT POSSIBLE for CYP2C9.
  Only 1 P/LP and 7 B/LB variants carry a cadd_phred.
  A likelihood ratio needs both classes in every band. With these counts the
  LR+ values above 

In [16]:
# Q5
# (a) VUS rows in the filtered set
V = G[G.label == "VUS"].copy()

# (b) map each VUS to the evidence strength of its calibrated band
if CALIBRATED:
    V["band"] = pd.cut(V[SCORE_COL], BANDS, include_lowest=True)
    V["evidence"] = V["band"].map(cal["evidence"]).astype(object)
    V["evidence"] = V["evidence"].fillna("not scorable: no score")
else:
    scored = V[SCORE_COL].notna()
    V["evidence"] = np.where(scored,
                             "none: calibration not possible for this gene",
                             "none: not scorable, no score exists")

def call_for(ev):
    if ev.startswith("none: not scorable"):
        return "VUS (unscored, tools are blind to it)"
    if "pathogenic" in ev and ev.startswith(("supporting", "moderate", "strong")):
        return "VUS, moved towards pathogenic"
    if "benign" in ev and ev.startswith(("supporting", "moderate", "strong")):
        return "VUS, moved towards benign"
    return "VUS (unchanged)"

V["provisional_call"] = V["evidence"].map(call_for)

# (c) counts
n_path = V.provisional_call.str.contains("towards pathogenic").sum()
n_ben  = V.provisional_call.str.contains("towards benign").sum()
n_blind = V.provisional_call.str.contains("unscored").sum()
n_stay = len(V) - n_path - n_ben
print(f"VUS in the filtered set: {len(V)}")
print(f"  towards pathogenic: {n_path}")
print(f"  towards benign:     {n_ben}")
print(f"  stay put:           {n_stay}  (of which unscored: {n_blind})")
print("\nprovisional_call counts:")
print(V.provisional_call.value_counts().to_string())

# (d) export
out_cols = ["rsid", "gene", "consequence", "aa_change", SCORE_COL, "gnomad_af",
            "evidence", "provisional_call"]
OUT = f"pgx_classified_{MY_GENE}.csv"
V[out_cols].to_csv(OUT, index=False)
print(f"\nwrote {OUT}  ({len(V)} rows)")
print(V[out_cols].head(10).to_string(index=False))

VUS in the filtered set: 22
  towards pathogenic: 0
  towards benign:     0
  stay put:           22  (of which unscored: 0)

provisional_call counts:
provisional_call
VUS (unchanged)    22

wrote pgx_classified_CYP2C9.csv  (22 rows)
        rsid   gene      consequence aa_change  cadd_phred  gnomad_af                                     evidence provisional_call
        None CYP2C9 missense_variant      C10Y       3.849        NaN none: calibration not possible for this gene  VUS (unchanged)
rs1210894054 CYP2C9 missense_variant     D468H      22.900   0.000004 none: calibration not possible for this gene  VUS (unchanged)
        None CYP2C9 missense_variant     E300V      26.500        NaN none: calibration not possible for this gene  VUS (unchanged)
rs2031306544 CYP2C9 missense_variant      L15I       9.711        NaN none: calibration not possible for this gene  VUS (unchanged)
 rs370407644 CYP2C9 missense_variant     E274Q      23.300   0.000007 none: calibration not possible for t

In [17]:
SKIP_ATLAS   = False
MAX_VARIANTS = 200

import os
ATLAS_STATUS = "not run"

def to_variant_string(hgvs):
    """Atlas wants 1-based GRCh38 chr:pos:ref>alt. It does NOT accept rsIDs."""
    m = re.match(r"(chr[\w]+):g\.(\d+)([ACGT])>([ACGT])", str(hgvs))
    return f"{m.group(1)}:{m.group(2)}:{m.group(3)}>{m.group(4)}" if m else None

if SKIP_ATLAS:
    print("SKIPPED. Disclose this in your report and answer Q6.2 from reasoning.")
    ATLAS_STATUS = "skipped"
else:
    try:
        from alphagenome.atlas import atlas
        from alphagenome.data import genome
        key = os.environ.get("ALPHAGENOME_API_KEY") or os.environ.get("ALPHA_GENOME_API_KEY")
        if not key:
            raise RuntimeError(
                "No API key in your environment. Did you relaunch Jupyter from a NEW\n"
                "  terminal after setting it? Check with:\n"
                "    import os; print(os.environ.get('ALPHAGENOME_API_KEY'))\n"
                "  Or set SKIP_ATLAS = True.")
        client = atlas.create(key, timeout=30)

        pri = G.assign(_nc=~G.consequence.fillna("").str.contains("missense")) \
               .sort_values("_nc", ascending=False).head(MAX_VARIANTS)
        rows, errs = [], 0
        for n, (_, r) in enumerate(pri.iterrows(), 1):
            vs = to_variant_string(r.hgvs_g)
            if not vs:
                continue
            try:
                res = client.query_variant(
                    genome.Variant.from_str(vs),
                    requested_scorers=["AVI_SCORE", "AVI_SCORE_FEATURE_IMPORTANCE"])
                ad = res["AVI_SCORE"]
                cdf = float(ad.layers["quantiles"][0, 0])
                fi = res["AVI_SCORE_FEATURE_IMPORTANCE"]
                rows.append({"hgvs_g": r.hgvs_g,
                             "avi_phred": round(-10*np.log10(max(1-cdf, 1e-12)), 2),
                             "top_feature": fi.var["name"].values[int(np.argmax(np.abs(fi.X[0])))]})
            except Exception:
                errs += 1
            if n % 50 == 0:
                print(f"  {n}/{len(pri)} queried", flush=True)

        avi = pd.DataFrame(rows)
        if len(avi):
            avi.to_csv(DATA / f"avi_{MY_GENE}.tsv", sep="\t", index=False)
            G = G.merge(avi, on="hgvs_g", how="left")
            G["avi"] = G.avi_phred
            ATLAS_STATUS = "queried live"
            print(f"\nscored {len(avi)} variants ({errs} failures)")
    except ImportError:
        print('alphagenome not installed. pip install "alphagenome>=0.9.0", then restart.')
        print("Or set SKIP_ATLAS = True.")
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

if "avi" not in G.columns: G["avi"] = np.nan
if "top_feature" not in G.columns: G["top_feature"] = np.nan
G["avi"] = pd.to_numeric(G["avi"], errors="coerce")

if G.avi.notna().sum() > 0:
    a = G[G.avi.notna()]
    cols = [c for c in ["rsid", "consequence", "avi_phred", "top_feature",
                        "revel", "cadd_phred", "label"] if c in a.columns]
    print(f"\nhighest-impact {MY_GENE} variants by AVI\n")
    print(a.sort_values("avi", ascending=False)[cols].head(10).to_string(index=False))
    print("\nwhat is driving the scores (read this BEFORE the numbers)")
    print(a.top_feature.value_counts().head(8).to_string())
else:
    nc = G[~G.consequence.fillna("").str.contains("missense")]
    print(f"\nnon-missense variants in {MY_GENE}, which is what the Atlas is for: {len(nc)}")
    print(nc.consequence.value_counts().to_string())

print(f"\n[Atlas status: {ATLAS_STATUS}]")


scored 36 variants (2 failures)

highest-impact CYP2C9 variants by AVI

        rsid      consequence  avi_phred     top_feature  revel  cadd_phred label
 rs370407644 missense_variant      23.41 MERGED_SPLICING  0.144       23.30   VUS
rs1226372739 missense_variant      22.20   ALPHAMISSENSE  0.633       26.30   VUS
        None missense_variant      22.03   ALPHAMISSENSE  0.320       26.50   VUS
  rs28371686 missense_variant      21.42   ALPHAMISSENSE  0.082       23.50  B/LB
rs1170075870 missense_variant      18.02   ALPHAMISSENSE  0.536       24.60   VUS
        None missense_variant      17.53   ALPHAMISSENSE  0.146       25.30   VUS
   rs2256871 missense_variant      16.82   ALPHAMISSENSE  0.105       23.80  B/LB
rs1210894054 missense_variant      16.50   ALPHAMISSENSE  0.159       22.90   VUS
rs1371272359 missense_variant      13.28   ALPHAMISSENSE  0.255       16.83   VUS
  rs28371685 missense_variant      13.21   ALPHAMISSENSE  0.222       27.50 other

what is driving the scor

In [18]:
d = G.copy()
coding = d.consequence.isin(["missense_variant", "synonymous_variant"])

d["flag_cadd_vs_revel"]  = (d.cadd_phred > 25) & (d.revel < 0.35)
d["flag_revel_vs_am"]    = (d.revel - d.alphamissense).abs() > 0.4
d["flag_scored_but_vus"] = (d.label == "VUS") & (d.revel > 0.7)
d["flag_common_but_plp"] = (d.gnomad_af > 0.01) & (d.label.isin(["P/LP", "other"]))
d["flag_no_tool_at_all"] = d[["revel", "alphamissense", "cadd_phred"]].isna().all(axis=1)
if d.avi.notna().any():
    d["flag_atlas_only"] = (d.avi > 20) & d.revel.isna() & d.alphamissense.isna()

flags = [c for c in d.columns if c.startswith("flag_")]
print(f"discordant variants in {MY_GENE}\n")
print(d[flags].sum().to_string())

hits = d[d[flags].any(axis=1)]
cols = [c for c in ["rsid", "note", "consequence", "consequence_all", "aa_change",
                    "cadd_phred", "revel", "alphamissense", "avi", "gnomad_af",
                    "label"] if c in d.columns]
print(f"\n{len(hits)} variants trip at least one rule\n")
print(hits[cols].head(15).to_string(index=False) if len(hits) else "  (none)")

if len(hits) == 0:
    print("\nNone. That is a legitimate result. Say so in your report and explain why")
    print("this gene's variants are concordant, rather than loosening the rules until")
    print("something fires.")

discordant variants in CYP2C9

flag_cadd_vs_revel     4
flag_revel_vs_am       2
flag_scored_but_vus    0
flag_common_but_plp    3
flag_no_tool_at_all    3
flag_atlas_only        0

11 variants trip at least one rule

        rsid        consequence                          consequence_all aa_change  cadd_phred  revel  alphamissense   avi  gnomad_af label
        None frameshift_variant frameshift_variant;splice_region_variant      None         NaN    NaN            NaN   NaN   0.003103  B/LB
  rs28371685   missense_variant                         missense_variant     R335W      27.500  0.222         0.2408 13.21   0.007015 other
        None   missense_variant                         missense_variant     E300V      26.500  0.320         0.6387 22.03        NaN   VUS
   rs1057910   missense_variant                         missense_variant     I359L      20.400  0.070         0.1487  9.72   0.049454 other
        None               None                                               None

In [19]:
none_tool = G[G[["cadd_phred", "revel", "alphamissense"]].isna().all(axis=1)]
cols = ["hgvs_g", "clinvar_id", "consequence", "consequence_all",
        "clinvar_sig", "label", "gnomad_af", "clinvar_stars"]
print(none_tool[cols].to_string(index=False))

             hgvs_g  clinvar_id        consequence                          consequence_all                 clinvar_sig label  gnomad_af  clinvar_stars
chr10:g.94949282del      285601 frameshift_variant frameshift_variant;splice_region_variant Benign/Likely benign; other  B/LB   0.003103              2
chr10:g.94942290C>C      163021               None                                                        drug response other        NaN              0
chr10:g.94981296A>A      177709               None                                                        drug response other        NaN              0
